## 工具的定义方式2：使用@tool装饰器(推荐)
使用@tool 装饰器修饰，可以自动将普通Python 函数转化为智能体可调用的工具。此方式最直接，代码量极少，非常适合快速验证想法或创建参数简单的工具。

## 1. 自定义工具描述：description

### 情况1：仅提供docstring信息

In [3]:
from jaraco.classes import properties
from langchain_core .utils .function_calling import convert_to_openai_tool
from langchain.tools import tool

@tool
def get_weather(city: str):
    """
    获取天气
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather', 'description': '获取天气', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


### 情况2：添加工具描述：description


In [8]:
from langchain_core .utils .function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

#有tool工具类的description，会优先按照这个description来解释说明

@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """
    获取当日天气

    Args:
        city: 城市

    Returns:
        返回当天的天气预报
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 情况3：解析docstring：parse_docstring

In [6]:
from langchain_core .utils .function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

#这个加上parse_docstring=True非常关键，不加的话。文档注释就不会正常
@tool(parse_docstring=True)
def get_weather(city: str, units: str = "celsius",include_forecast: bool = False ) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报

    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来5日的天气预报

    Returns:
        返回当天的天气预报，可选择未来5天的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温：{temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

rprint(convert_to_openai_tool(get_weather))


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来5日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2. 更改工具名称：name_or_callable

In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool

#注意：更改名称name_or_callable可以省掉。直接用名字也可以
@tool(name_or_callable="getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))
print("=" * 100)

{'type': 'function', 'function': {'name': 'getWeather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


## 3.  自定义args_schema
### 方式1：使用Pydantic模型定义
当工具的参数变得复杂，需要枚举值、范围限制 或 更复杂的业务逻辑验证时，Pydantic 模型是理想的选择，提供强大的类型检查和数据验证。

使用Pydantic 的主要优势在于能够精确控制工具参数的格式和验证规则，让大模型更准确地理解如何调用工具。

### 3.1 pydantic类型的定义


BaseModel基类

通过继承核心基类BaseModel 定义数据模型，从而声明字段结构、类型约束、默认值以0及校验规则。

In [18]:
from pydantic import BaseModel

class WeatherInput(BaseModel):
    city: str

print(WeatherInput(city="北京"))


city='北京'


**Field**

Field()：用来“定制字段”的函数，可用于设置默认值、描述等。

In [20]:
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city: str = Field(
        default="北京",
        description="城市"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )

print(WeatherInput())

city='北京' include_forecast=False


**Literal**

可以使用Literal类型限定参数为固定选项。

In [22]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    city: str = Field(
        default="北京",
        description="城市"
    )
    unit : Literal ["celsius ", "fahrenheit"] = Field(
        default="celsius ",
        description="气温单位"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )

print(WeatherInput())

city='北京' unit='celsius ' include_forecast=False


### 3.2：使用Pydantic定义args_schema

In [27]:
from langchain_core .utils .function_calling import convert_to_openai_tool
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    city: str = Field(
        default="北京",
        description="城市"
    )
    unit : Literal ["celsius ", "fahrenheit"] = Field(
        default="celsius ",
        description="气温单位"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )

@tool(args_schema=WeatherInput)
def get_weather(city: str, units: str = "celsius",include_forecast: bool = False ) -> str:
    """获取当日天气，可选未来五日天气预报"""
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温：{temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

convert_to_openai_tool(get_weather)

{'type': 'function',
 'function': {'name': 'get_weather',
  'description': '获取当日天气，可选未来五日天气预报',
  'parameters': {'properties': {'city': {'default': '北京',
     'description': '城市',
     'type': 'string'},
    'unit': {'default': 'celsius ',
     'description': '气温单位',
     'enum': ['celsius ', 'fahrenheit'],
     'type': 'string'},
    'include_forecast': {'default': False,
     'description': '是否包含未来五日天气预报',
     'type': 'boolean'}},
   'type': 'object'}}}

### 方式2：使用Json Schema定义

In [31]:
from langchain_core .utils .function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

#定义一个json的变量
weather_json_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"},
    },
    "required": ["city", "units", "include_forecast"]
}

@tool(args_schema=weather_json_schema)
def get_weather(city: str, units: str = "celsius",include_forecast: bool = False ) -> str:
    """获取当日天气，可选未来五日天气预报"""
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温：{temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选未来五日天气预报',
        'parameters': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string'},
                'units': {'type': 'string'},
                'include_forecast': {'type': 'boolean'}
            },
            'required': ['city', 'units', 'include_forecast']
        }
    }
}